In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from shared.make_envs import make_procgen
from ppo_cleanrl.config import StrConfig
envs = make_procgen(StrConfig())
tmp = envs.reset()








In [ ]:
def scan_step(carry, t):
    obs, done, rng = carry
    actions = np.zeros((64), dtype = np.uint8)
    next_obs, rewards, next_done, infos = envs.step(actions)
    next_obs = jnp.array(next_obs, dtype = jnp.uint8)
    rewards = jnp.array(rewards, dtype = jnp.float32)
    next_done = jnp.array(next_done, dtype = jnp.uint8)
    output = (obs, actions, rewards, done)
    new_carry = (next_obs, next_done, rng)
    return new_carry, output

obs = envs.reset()
done = jnp.zeros((64), dtype = jnp.uint8)
T = 256
rng = jax.random.PRNGKey(0)

%timeit (_, _, _), (obs_traj, actions_traj, rewards_traj, dones_traj) = jax.lax.scan(scan_step, (obs, done, rng), None, length = T)

print(obs_traj.shape)
print(actions_traj.shape)
print(rewards_traj.shape)
print(dones_traj.shape)






In [ ]:
from ppo_cleanrl.episode import make_episode
episode = make_episode(
    obs_shape = (64, 64, 3),
    act_dim = (),
    num_steps = 256,
    num_envs = 64,
)
print(episode.actions.shape)

In [ ]:

next_obs = envs.reset()
next_done = np.zeros((64), dtype = np.uint8)


In [ ]:
%%timeit
next_obs = envs.reset()
next_done = np.zeros((64), dtype = np.uint8)
for i in range(256):
    actions = np.zeros((64), dtype = np.uint8)
    episode.obs[i] = next_obs
    episode.dones[i] = next_done
    next_obs, rewards, next_done, infos = envs.step(actions)
    episode.actions[i] = actions
    episode.rewards[i] = rewards

jax.device_put(episode.obs, jax.devices('gpu')[0])
jax.device_put(episode.dones, jax.devices('gpu')[0])
jax.device_put(episode.actions, jax.devices('gpu')[0])
jax.device_put(episode.rewards, jax.devices('gpu')[0])